In [ ]:
# Do Not Modified. Generated by iantirta.com
import os
os.environ['IANTIRTA_API_KEY'] = 'fb1b8a4dafbe1b197b5e2f2a9e38f68e132aed3af691fca1a5f2314ea892d38c'

iantirta_url = "http://localhost"
worker_name = "Kaggle Worker `crows.nony1@gmail.com`"
worker_provider = "kaggle"

!pip install -q --upgrade --force-reinstall --no-cache-dir "git+https://github.com/yancovenant/karaokeplus.git"

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

DRIVE_USER_TOKEN: dict = {"token": "ya29.a0AX07CmtkGfz8HVXINDqRgedyD7IDMcwuMziloC9L3Vum1WQ70MeX6HiiCddw2Qk71-EQnDpIkyEMb5u0fMO1x3LBamIBB_1y51KOIHc3U53U2RfbFCn0V7dZsusJZce8kqWbh2VjChJGX54Ioqu9Pa6zfX4tcSuyPBshDbCPo4lAnhESJZNKnhsUj0ZbSlyyjKJ02WUaCgYKAV8SARUSFQHGX2MiBLf_N5hw5ahifaT7LdskUA0206", "refresh_token": "1//0gO7J6xNJWm7DCgYIARAAGBASNwF-L9Ir3wachxCkxfHiFRbttQ5vu5hYMh4LT5TfQfKhJ3MubhSaYT_Nx84Gm2e1CX10NBl_S6U", "token_uri": "https://oauth2.googleapis.com/token", "client_id": "294185921543-1ptm5pr2j3iqv10vlmj0vn16dp8m779q.apps.googleusercontent.com", "client_secret": "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", "scopes": ["https://www.googleapis.com/auth/drive"], "universe_domain": "googleapis.com", "account": "", "expiry": "2026-09-22T15:05:10.963124Z"}
SHARED_ROOT_FOLDER_ID = "1"
print("ok")

In [ ]:
#@title Karaoke Plus
from __future__ import annotations

import argparse
import concurrent.futures
import glob
import json
import logging
import os
import subprocess
import typing as t
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path

import kplus
import requests
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from kplus import config, env
from kplus.pipelines import (
    align,
    align2ref,
    detect_audio_activity,
    ensure_file,
    transcribe,
)
from kplus.pipelines.asr.hf import HFModel
from kplus.pipelines.separate import BaseSeparator, DemucsSeparator
from kplus.pipelines.utils import ASRResult
from kplus.tools.render import Render

# Logging Setup
opt = argparse.Namespace(log_level="debug")
config.parse_config(opt, setup_logging=True)
logger = logging.getLogger(__name__)

# Secret getter
def get_secret(key: str, *args, **kwargs):
    """ Return a secret defaulting to os.environ.
    """
    try:
        if kplus.env.is_kaggle:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key, *args, *kwargs)
        elif kplus.env.is_colab:
            from google.colab import userdata
            return userdata.get(key, *args, *kwargs)
    except:
        pass
    return os.environ.get(key, *args, **kwargs)


TaskStatus: t.TypeAlias = t.Literal["waiting", "processing", "completed", "error"]
TaskType: t.TypeAlias = t.Literal["basic", "plus"]

if t.TYPE_CHECKING:
    from kplus.pipelines.utils import ASRResult, AudioSegment
    from kplus.pipelines.asr.hf.mixin import ASRMixin


@dataclass(slots=True)
class Task:
    """ Hold Task Type """
    id: int
    title: str
    artist: str
    duration: float
    lyrics: str
    status: TaskStatus
    url: str
    karaoke_type: TaskType

    # Processing helper
    audiopath: str | None = field(init=False, default=...)
    instpath: str | None = field(init=False, default=...)
    samplerate: int | None = field(init=False, default=...)
    audiosegments: AudioSegment | None = field(init=False, default=...)
    result: ASRResult | None = field(init=False, default=...)

    error: str | None = field(init=False, default=...)
    log: str | None = field(init=False, default=...)
    videopath: str | None = field(init=False, default=...)
    karaokepath: str | None = field(init=False, default=...)

    # Complete
    download_url: str | None = field(init=False, default=...)
    drive_folder_id: str | None = field(init=False, default=...)
    drive_file_id: str | None = field(init=False, default=...)

    @classmethod
    def from_dict(cls, data: dict) -> Task:
        return cls(
            id=data["id"],
            title=data["title"],
            artist=data["artist"],
            duration=data["duration"],
            lyrics=data["lyrics"],
            status=data["status"],
            url=data["url"],
            karaoke_type=data["karaoke_type"],
        )

    def update_status(self, status: str, message: str):
        """ This should be update error
        """
        self.status = status
        self.error = message


tasks: list[Task] = []


class APIError(Exception):
    """ Handle API Error """


class IantirtaAPI:
    """ API Needed for connection to iantirta.com. """
    def __init__(self):
        self.api_token = get_secret("IANTIRTA_API_KEY")

    def post(self, url: str, params: dict):
        data = {
            "params": {
                "worker_name": worker_name,
                "worker_provider": worker_provider,
                "access_token": self.api_token,
            }
        }
        if "params" in params:
            data["params"].update(params["params"])
        else:
            data["params"].update(params)
        headers = {
            "Authorization": f"bearer {self.api_token}",
        }
        try:
            res = requests.post(url, headers=headers, json=data, allow_redirects=False)
            res.raise_for_status()
            if res.is_redirect:
                new_url = res.headers["Location"]
                res = requests.post(new_url, headers=headers, json=data)
                res.raise_for_status()
            
            res = res.json()
            if res.get("error", False):
                raise APIError("API Error:", res)
            else:
                return res["result"]
        except Exception:
            raise

    def get_task(self, status_to_fetch: TaskStatus) -> list[Task]:
        api_url = f"{iantirta_url}/karaoke/task"
        params = {"params": {
            "status_to_fetch": status_to_fetch,
        },}
        res = self.post(api_url, params)

        global tasks
        tasks = [Task.from_dict(r) for r in res]
        return tasks

    def update_task(self, tasks: list[Task]) -> bool:
        api_url = f"{iantirta_url}/karaoke/task/update"
        params = {"params": {
            "datas": [
                {
                    "status": task.status,
                    "download_url": task.download_url,
                    "drive_folder_id": task.drive_folder_id,
                    "drive_file_id": task.drive_file_id,
                    "error": task.error,
                    "log": task.log,
                }
                for task in tasks
            ]
        }}
        self.post(api_url, params)

    def post_worker(self, tasks: list[Task]) -> bool:
        api_url = f"{iantirta_url}/karaoke/worker/update"
        params = {"params": {
            "karaoke_ids": [task.id for task in tasks]
        }}
        self.post(api_url, params)
        return True

class DriveAPI:
    """ API Needed for connection to google.drive. """
    SCOPES = ['https://www.googleapis.com/auth/drive']  # noqa: RUF012
    SHARED_ROOT_FOLDER_ID = "18kRj0gVXeDhOWwdq1GnuBTjBVWqvYKJN"

    def __init__(self):
        self.creds = self._get_creds()

    def _get_creds(self):
        creds = None
        search_paths = [
            'user_token.json',
            '/kaggle/input/jazpiper-automation/user_token.json'
        ]
        # Kaggle
        if os.path.exists('/kaggle/input/'):
            found_files = glob.glob('/kaggle/input/**/*user_token.json', recursive=True)
            if found_files:
                search_paths.append(found_files[0])

        user_token_path = "user_token.json"
        for path in search_paths:
            if os.path.exists(path):
                creds = Credentials.from_authorized_user_file(path, self.SCOPES)
                user_token_path = path
                break
        
        #
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            elif os.path.exists("creds.json"):
                flow = InstalledAppFlow.from_client_secrets_file(
                    'creds.json',
                    self.SCOPES,
                    redirect_uri='urn:ietf:wg:oauth:2.0:oob'
                )
                auth_url, _ = flow.authorization_url(prompt='consent')
                print(f'Please go to this URL: {auth_url}')
                code = input('Enter the Authorization Code: ')
                flow.fetch_token(code=code)
                creds = flow.credentials
            else:
                raise RuntimeError("Cannot continue as drive api credentials isn't exists")
            
            with open('user_token.json', 'w') as token:
                token.write(creds.to_json())
                
        return creds
    
    def upload_drive(self, task: Task):
        service = build('drive', 'v3', credentials=self.creds)

        def get_or_create(folder_name: str, parent_id: str | None = None) -> str:
            query = (
                f"name='{folder_name}' "
                "and mimeType='application/vnd.google-apps.folder' "
                "and trashed=false"
            )
            if parent_id:
                query += f" and '{parent_id}' in parents"
            
            response = service.files().list(
                q=query,
                spaces='drive',
                fields='files(id, name)',
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()

            if files := response.get('files', []):
                return files[0]['id']

            file_metadata = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder'}

            if parent_id:
                file_metadata['parents'] = [parent_id]
            
            folder = service.files().create(
                body=file_metadata,
                fields='id',
                supportsAllDrives=True
            ).execute()

            return folder.get('id')

        outpath = Path(task.karaokepath)
        artist_folder_id = get_or_create(
            task.artist,
            self.SHARED_ROOT_FOLDER_ID
        )
        file_metadata = {'name': outpath.name, 'parents': [artist_folder_id]}

        media = MediaFileUpload(str(outpath), mimetype='video/x-matroska', resumable=True)
        request = service.files().create(
            body=file_metadata,
            media_body=media,
            fields='id, webViewLink, webContentLink',
            supportsAllDrives=True,
        )

        response = None
        while response is None:
            status, response = request.next_chunk()
            if status:
                logger.debug(f"Upload progress for Task {task.id}: {int(status.progress() * 100)}%")
        
        file_id = response.get('id')
        download_url = response.get('webContentLink')

        service.permissions().create(
            fileId=file_id, body={'type': 'anyone', 'role': 'reader'}, fields='id'
        ).execute()

        logger.info(f"Successfully uploaded Task {task.id} to Drive. File ID: {file_id}")
        outpath.unlink(missing_ok=True)

        task.download_url = download_url
        task.drive_folder_id = artist_folder_id
        task.drive_file_id = file_id

        return task

class CloudflareAPI:
    """ API Needed for connection to cloudflare storage. """

class KaraokeWorker:
    def __init__(self):
        self.cookiefile = "cookies.txt"
        self.iantirta_api = IantirtaAPI()
        self.iantirta_api.get_task("waiting")
        self.drive_api = DriveAPI()

    def _run_render(
        self,
        task: Task,
        instrument_path: str,
        result: ASRResult | None = None
    ) -> None:
        task.karaokepath = Render(with_ass=task.karaoke_type == 'plus').render(
            video_filepath=task.videopath,
            inst_path=instrument_path,
            duration=task.duration,
            result=result,
            output_path=None,
        )
        task.status = "completed"
        self.drive_api.upload_drive(task)
        
        return task

    def _run(
        self,
        task: Task,
        separator: DemucsSeparator,
        executor: ThreadPoolExecutor,
    ) -> concurrent.futures.Future | None:
        # Normalize to wav
        inputpath = Path(task.videopath).expanduser().resolve()
        mixpath = inputpath.stem + ".wav"
        subprocess.run([
            "ffmpeg", "-y", "-i",
            str(inputpath), "-vn",
            "-ar", str(separator.sr),
            "-ac", str(separator.ac),
            str(mixpath)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Run separation
        sep_result = separator.separate(mixpath)
        instrument_path = sep_result.inst_path

        if os.path.exists(mixpath):
            os.unlink(mixpath)
        
        if task.karaoke_type == "basic":
            # Imidiately pass it to render & upload
            return executor.submit(
                self._run_render,
                task,
                instrument_path,
            )
        elif task.karaoke_type == "plus":
            # Pass here since it needs to be run separately.
            task.audiopath = sep_result.vocs_path
            task.instpath = instrument_path
            task.samplerate = sep_result.sr
            return None
        else:
            raise RuntimeError(
                f"Karaoke Type `{task.karaoke_type}` "
                "isn't supported yet."
            )
    
    def run(self) -> None:
        """ Pipeline
            * basic_tasks: (download, separate, render)
            * plust_tasks: (download, separate, transcribe, render)
        """
        logger.debug(f"Processing {len(tasks)}...")
        self.iantirta_api.post_worker(tasks)

        render_executor = ThreadPoolExecutor(max_workers=3)
        render_futures: dict[concurrent.futures.Future, Task] = {}

        separator = BaseSeparator.from_options(
            demucs="mdx_extra_q",
            overlap=0.75,
            segment=200,
            shifts=1,
            num_workers=0,
        )

        with ThreadPoolExecutor(max_workers=5) as executor:
            iter_task = iter(tasks)
            active_futures: dict[t.Any, Task] = {}
            for _ in range(5):
                if task := next(iter_task, None):
                    future = executor.submit(
                        ensure_file,
                        inputpath=task.url,
                        external_id=task.id,
                        no_lyrics=task.karaoke_type == "basic",
                        cookiefile=self.cookiefile,
                    )
                    active_futures[future] = task
            while active_futures:
                done, _ = concurrent.futures.wait(
                    active_futures.keys(),
                    return_when=concurrent.futures.FIRST_COMPLETED
                )
                for future in done:
                    task = active_futures.pop(future)
                    if next_task := next(iter_task, None):
                        new_future = executor.submit(
                            ensure_file,
                            inputpath=next_task.url,
                            external_id=next_task.id,
                            no_lyrics=next_task.karaoke_type == "basic",
                            cookiefile=self.cookiefile,
                        )
                        active_futures[new_future] = next_task
                    try:
                        info = future.result()
                        assert info.duration == task.duration, (
                            "Duration Missmatch between "
                            "server side and worker side, "
                            f"{info.duration} | {task.duration}"
                        )
                        assert info.title == task.title, (
                            "Title Missmatch between "
                            "server side and worker side, "
                            f"{info.title} | {task.title}"
                        )
                        assert info.artist == task.artist, (
                            "Artist Missmatch between "
                            "server side and worker side, "
                            f"{info.artist} | {task.artist}"
                        )
                        task.videopath = info.filepath
                        if r_future := self._run(task, separator=separator, executor=render_executor):
                            render_futures[r_future] = task
                    except Exception as e:
                        logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                        task.update_status("failed", str(e))

        del separator.model, separator
        env.clean()

        # Audio Detection
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    audio_result = detect_audio_activity(
                        audio=task.audiopath,
                        sr=task.samplerate,
                        no_show=True,
                        resample=False,
                        use_html=False,
                    )
                    task.audiosegments = audio_result.segments
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))

        # ASR
        transcriber: ASRMixin = HFModel.from_pretrained(
            "Qwen/Qwen3-ASR-1.7B-hf",
            max_inference_batch_size=1,
            num_beams=4,
        )
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    asr_result = transcriber.transcribe(
                        audio=task.audiopath,
                        audiosegments=task.audiosegments,
                        contexts=task.lyrics,
                        languages=None,
                        return_timestamps=True,
                    )
                    task.result = asr_result
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))

        del transcriber.model, transcriber
        env.clean()

        # into Lyrics
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    ref_result, new_audiosegments = align2ref(
                        task.result,
                        task.lyrics,
                        task.audiosegments,
                        raise_if_not_reliable=False
                    )
                    task.result = ref_result
                    task.audiosegments = new_audiosegments
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))

        # Alignment
        aligner: ASRMixin = HFModel.from_pretrained("facebook/mms-1b-all",)
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    align_results = aligner.align(
                        audio=task.audiopath,
                        audiosegments=task.audiosegments,
                        hypothesis=task.result.texts,
                        languages=None,
                    )
                    align_results.to_line_idx(task.lyrics)
                    task.result = align_results.populate_ass()
                    r_future = render_executor.submit(
                        self._run_render,
                        task,
                        task.instpath,
                        task.result,
                    )
                    render_futures[r_future] = task
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))
        del aligner.model, aligner.processor, aligner
        env.clean()
        
        for future in as_completed(render_futures):
            task = render_futures[future]
            try:
                future.result()  # This will raise exceptions if the render/upload failed
            except Exception as e:
                logger.error(f"Render/Upload Failed for Task {task.id}: {e}", exc_info=True)
                task.update_status("error", str(e))

        self.iantirta_api.update_task(tasks)

        render_executor.shutdown()
        
        logger.info("Complete: All tasks processed through pipeline.")
        logger.info("complete")

if DryRun:
    print("== Test API ==")
    iantirta_api = IantirtaAPI()
    iantirta_api.get_task("waiting")
    for task in tasks:
        print(task.karaoke_type)
    iantirta_api.post_worker(tasks)
else:
    KaraokeWorker().run()